In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

## Install libraries

#install -q youtube-transcript-api 

In [25]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings

## Step 1a - Indexing (Document Ingestion)

In [18]:
# video_id = "ysPbXH0LpIE" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id='ysPbXH0LpIE')

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Hi everyone. Thank you for joining us today for 
prompting 101. Uh my name is Hannah. I'm part   of the applied AI team here at Anthropic. And 
with me is Christian, also part of the applied   AI team. And what we're going to do today is 
take you through a little bit of prompting best   practices. And we're going to use a real world 
scenario and build up a prompt together. Uh so   a little bit about what prompt engineering is. uh 
prompt engineering. You're all probably a little   bit familiar with this. This is the way that we 
communicate with a language model and try to get   it to do what we want. So, this is the practice of 
writing clear instructions for the model, giving   the model the context that it needs to complete 
the task, and thinking through how we want to   arrange that information in order to get the 
best result. Um, so there's a lot of detail here,   a lot of different ways you might want to think 
about building out a prompt. Um, and as always,   the best way to

In [19]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="Hi everyone. Thank you for joining us today for\xa0\nprompting 101. Uh my name is Hannah. I'm part\xa0\xa0", start=5.04, duration=5.44), FetchedTranscriptSnippet(text='of the applied AI team here at Anthropic. And\xa0\nwith me is Christian, also part of the applied\xa0\xa0', start=10.48, duration=4.8), FetchedTranscriptSnippet(text="AI team. And what we're going to do today is\xa0\ntake you through a little bit of prompting best\xa0\xa0", start=15.28, duration=4.4), FetchedTranscriptSnippet(text="practices. And we're going to use a real world\xa0\nscenario and build up a prompt together. Uh so\xa0\xa0", start=19.68, duration=5.76), FetchedTranscriptSnippet(text="a little bit about what prompt engineering is. uh\xa0\nprompt engineering. You're all probably a little\xa0\xa0", start=25.44, duration=5.04), FetchedTranscriptSnippet(text='bit familiar with this. This is the way that we\xa0\ncommunicate with a language model and try t

## Step 1b - Indexing (Text Splitting)

In [20]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [21]:
len(chunks)

36

In [23]:
chunks[10]

Document(metadata={}, page_content="report form, and it's just 17 different checkboxes\xa0\xa0 going through what actually happened. You\xa0\ncan see there's a vehicle A and vehicle B,\xa0\xa0 both on the left and right hand side. And the main\xa0\npurpose of this is that we want to make sure that\xa0\xa0 Claude can understand this manually generated data\xa0\nto assess what's actually going on. And that is\xa0\xa0 uh corroborated by if I navigate back here to this\xa0\nsketch that we can highlight here as well. In this\xa0\xa0 case, the form is just a different um data point\xa0\nfor the same scenario. Um and in this case here, I\xa0\xa0 want to bake in more information into our version\xa0\ntwo. Uh and by doing so, I'm actually elaborating\xa0\xa0 a lot more on what's going on. So, you can see\xa0\nhere I'm specifying that uh this AI assistant is\xa0\xa0 supposed to help a human's claim claims adjuster\xa0\nthat's reviewing car accident report forms in\xa0\xa0 Swedish as well. Um, yo

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [26]:
# Use Together AI endpoint
embeddings = OpenAIEmbeddings(
    model="intfloat/multilingual-e5-large-instruct",  # good embedding model
    openai_api_key="tgp_v1_--nKOPx_Rtr2sAiSOtvpafrbcvAjyajARRfAyZ8G_Bo",
    openai_api_base="https://api.together.xyz/v1"
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
vector_store.persist()

/var/folders/gt/nwfffcg538j_681fjbgf3l780000gn/T/ipykernel_4035/1014643419.py:12: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [28]:
data = vector_store.get()
print(data)

{'ids': ['6c0f0871-ce3b-42cf-9b6a-121e28920d0e', '8c7d91a6-efea-4660-a3c3-c4da2ceac358', '01460a2e-1c7e-42b7-b2e7-7aed642dc084', '2347a9aa-5914-4213-861a-89db71c40075', 'c50e9c56-5efc-45a5-bcbe-4d1e1f46b0b3', 'ece79cf2-80f6-4a12-b17e-e00d52de8e7c', '0a1def23-8512-4ff2-9b35-7cda0afbb7a7', 'bd0eef58-57dc-4d22-b362-d6d2d0a7cd2a', 'bdc8605d-5620-4794-a682-2f82a4aa31e6', 'ac591ca8-5f9e-41c0-ba25-6900e53ed11e', 'fd5df9ce-6604-4a6f-85b7-ed9dda07da8a', 'c0b056d8-6a67-4f20-8a19-56ec9600e02f', '657e5d02-85af-4dcb-aa65-d28341a7b0a1', '10393d34-3c22-4772-bde2-6a1f7c51005a', '140439f6-c4d1-432c-81a0-8f4678fa8f95', 'a1dee3fb-5ce0-4fff-b44b-91e0529d677d', 'bbeba0bf-093c-48cd-860d-5d396f6d81bc', '77b56c77-7952-4072-9e26-8d9e1460dc7c', 'abab0ca7-4a5c-475f-81c0-8c753ffeee67', '2ecee85c-56b8-4fbd-8244-5287974c1ad9', '6946ec68-5853-47e3-8eb0-f7f5dfa77d02', '2f77ef1d-bfac-437e-8b35-906be14d1837', 'a029aa6b-3b91-4fb5-9e73-fc51433fa200', '3786df8e-0f56-4d48-ab23-19dc4fb5c89d', 'd3698205-cd7b-4766-87e6-5f5153

In [29]:
ids = vector_store.get()["ids"]
print(ids)

['6c0f0871-ce3b-42cf-9b6a-121e28920d0e', '8c7d91a6-efea-4660-a3c3-c4da2ceac358', '01460a2e-1c7e-42b7-b2e7-7aed642dc084', '2347a9aa-5914-4213-861a-89db71c40075', 'c50e9c56-5efc-45a5-bcbe-4d1e1f46b0b3', 'ece79cf2-80f6-4a12-b17e-e00d52de8e7c', '0a1def23-8512-4ff2-9b35-7cda0afbb7a7', 'bd0eef58-57dc-4d22-b362-d6d2d0a7cd2a', 'bdc8605d-5620-4794-a682-2f82a4aa31e6', 'ac591ca8-5f9e-41c0-ba25-6900e53ed11e', 'fd5df9ce-6604-4a6f-85b7-ed9dda07da8a', 'c0b056d8-6a67-4f20-8a19-56ec9600e02f', '657e5d02-85af-4dcb-aa65-d28341a7b0a1', '10393d34-3c22-4772-bde2-6a1f7c51005a', '140439f6-c4d1-432c-81a0-8f4678fa8f95', 'a1dee3fb-5ce0-4fff-b44b-91e0529d677d', 'bbeba0bf-093c-48cd-860d-5d396f6d81bc', '77b56c77-7952-4072-9e26-8d9e1460dc7c', 'abab0ca7-4a5c-475f-81c0-8c753ffeee67', '2ecee85c-56b8-4fbd-8244-5287974c1ad9', '6946ec68-5853-47e3-8eb0-f7f5dfa77d02', '2f77ef1d-bfac-437e-8b35-906be14d1837', 'a029aa6b-3b91-4fb5-9e73-fc51433fa200', '3786df8e-0f56-4d48-ab23-19dc4fb5c89d', 'd3698205-cd7b-4766-87e6-5f5153779a8b',

In [31]:
results = vector_store.similarity_search("what is prompt engineering?", k=3)
print(results)

[Document(metadata={}, page_content="helpful to building up uh building up your prompt\xa0\xa0 and making it work better. Exactly. I think um\xa0\none thing that we really highlight is examples.\xa0\xa0 I think examples or few shot is a mechanism that\xa0\nreally is powerful in steering cloud. So you can\xa0\xa0 imagine this um in in quite a non-trivial way as\xa0\nwell. So imagine you have scenarios, situations,\xa0\xa0 even in this case concrete accidents that have\xa0\nhappened that are um tricky for claw to get right.\xa0\xa0 But you with your human intuition and your human\xa0\nlabel data um is able to actually get to the right\xa0\xa0 conclusion. Then you can bake that information\xa0\ninto the system problem itself by having clear-cut\xa0\xa0 examples of a the data that that it's supposed\xa0\nto look at. So you can have visual examples.\xa0\xa0 you can just base 64 encode a a an image and have\xa0\nthat as part of the data that you're passing along\xa0\xa0 into the examples and

## Step 2 - Retrieval

In [32]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [33]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x136b1b510>, search_kwargs={'k': 4})

In [34]:
retriever.invoke('What is deepmind')

[Document(metadata={}, page_content="helpful to building up uh building up your prompt\xa0\xa0 and making it work better. Exactly. I think um\xa0\none thing that we really highlight is examples.\xa0\xa0 I think examples or few shot is a mechanism that\xa0\nreally is powerful in steering cloud. So you can\xa0\xa0 imagine this um in in quite a non-trivial way as\xa0\nwell. So imagine you have scenarios, situations,\xa0\xa0 even in this case concrete accidents that have\xa0\nhappened that are um tricky for claw to get right.\xa0\xa0 But you with your human intuition and your human\xa0\nlabel data um is able to actually get to the right\xa0\xa0 conclusion. Then you can bake that information\xa0\ninto the system problem itself by having clear-cut\xa0\xa0 examples of a the data that that it's supposed\xa0\nto look at. So you can have visual examples.\xa0\xa0 you can just base 64 encode a a an image and have\xa0\nthat as part of the data that you're passing along\xa0\xa0 into the examples and

## Step 3 - Augmentation

In [35]:
# Cell: Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2, groq_api_key=groq_api_key)

In [36]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [43]:
question          = "is the topic of prompt engineering discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [44]:
retrieved_docs

[Document(metadata={}, page_content="context and the role. We're just adding this\xa0\xa0 detailed list of tasks. And this is how we want\xa0\nClaude to go about analyzing this. And a really\xa0\xa0 key thing that we found here as we were building\xa0\nthis demo and when we were working on the customer\xa0\xa0 example is that the order in which Claude analyzes\xa0\nthis information is very important. And this is\xa0\xa0 analogous to way you might think about doing this.\xa0\nIf you were a human, you would probably not look\xa0\xa0 at the drawing first and try to understand what\xa0\nwas going on, right? It's pretty unclear. It's\xa0\xa0 a bunch of boxes and lines. We don't really know\xa0\nwhat that drawing is supposed to mean without any\xa0\xa0 additional context. But if we have the form and we\xa0\ncan read the form first and understand that we're\xa0\xa0 talking about a car accident and that we're seeing\xa0\nsome checkboxes that indicate what vehicles we're\xa0\xa0 doing at certai

In [45]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"context and the role. We're just adding this\xa0\xa0 detailed list of tasks. And this is how we want\xa0\nClaude to go about analyzing this. And a really\xa0\xa0 key thing that we found here as we were building\xa0\nthis demo and when we were working on the customer\xa0\xa0 example is that the order in which Claude analyzes\xa0\nthis information is very important. And this is\xa0\xa0 analogous to way you might think about doing this.\xa0\nIf you were a human, you would probably not look\xa0\xa0 at the drawing first and try to understand what\xa0\nwas going on, right? It's pretty unclear. It's\xa0\xa0 a bunch of boxes and lines. We don't really know\xa0\nwhat that drawing is supposed to mean without any\xa0\xa0 additional context. But if we have the form and we\xa0\ncan read the form first and understand that we're\xa0\xa0 talking about a car accident and that we're seeing\xa0\nsome checkboxes that indicate what vehicles we're\xa0\xa0 doing at certain times, then we know a little\n\nhe

In [46]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [47]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      context and the role. We're just adding this\xa0\xa0 detailed list of tasks. And this is how we want\xa0\nClaude to go about analyzing this. And a really\xa0\xa0 key thing that we found here as we were building\xa0\nthis demo and when we were working on the customer\xa0\xa0 example is that the order in which Claude analyzes\xa0\nthis information is very important. And this is\xa0\xa0 analogous to way you might think about doing this.\xa0\nIf you were a human, you would probably not look\xa0\xa0 at the drawing first and try to understand what\xa0\nwas going on, right? It's pretty unclear. It's\xa0\xa0 a bunch of boxes and lines. We don't really know\xa0\nwhat that drawing is supposed to mean without any\xa0\xa0 additional context. But if we have the form and we\xa0\ncan read the form first and under

## Step 4 - Generation

In [48]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of prompt engineering is discussed in this video. Specifically, it was mentioned that baking in examples into the system prompt is a key aspect of prompt engineering, and that it's an empirical science of pushing the limits of the application and getting feedback to improve it. The speaker also emphasized the importance of using XML structure in the prompt.


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

'The video features a conversation with Demas, who discusses the need for deeper and simpler explanations in physics, particularly in relation to consciousness, life, and gravity. He emphasizes the limitations of the current standard model of physics and the importance of exploring more fundamental explanations. Additionally, he talks about advancements in fusion research, specifically how they have managed to hold plasma in specific shapes for extended periods, which is a significant step in fusion energy production. The conversation touches on the potential of AI in modeling quantum mechanical behavior, particularly regarding electrons. The discussion concludes with a quote about the nature of computer science.'